# RNA — Clasificación de Cáncer de Piel
## TP Cuatrimestral — Redes Neuronales Artificiales — UTN FRLP 2026

**Dataset**: HAM10000 — Human Against Machine with 10000 training images  
**Modelo**: Perceptrón Multicapa (MLP) con Backpropagation  
**Librería**: scikit-learn  

### Clases del problema

| Label | Código | Diagnóstico                          |
|-------|--------|--------------------------------------|
| 0     | akiec  | Queratosis actínica / intraepitelial |
| 1     | bcc    | Carcinoma basocelular                |
| 2     | bkl    | Queratosis benigna                   |
| 3     | df     | Dermatofibroma                       |
| 4     | nv     | Nevos melanocíticos (lunares)        |
| 5     | vasc   | Lesiones vasculares                  |
| 6     | mel    | Melanoma                             |

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

print('Librerías importadas correctamente.')

## 2. Carga y Exploración del Dataset

In [ ]:
# Carga del dataset (imágenes 8x8 en escala de grises — 64 features + label)
df = pd.read_csv('archive/hmnist_8_8_L.csv')

LABEL_NAMES_SHORT = {
    0: 'akiec', 1: 'bcc', 2: 'bkl',
    3: 'df',    4: 'nv',  5: 'vasc', 6: 'mel'
}
LABEL_FULL = {
    0: 'Queratosis actínica',
    1: 'Carcinoma basocelular',
    2: 'Queratosis benigna',
    3: 'Dermatofibroma',
    4: 'Nevo melanocítico',
    5: 'Lesión vascular',
    6: 'Melanoma'
}

print(f'Dimensiones: {df.shape[0]} muestras x {df.shape[1]} columnas')
print(f'Features de entrada: {df.shape[1]-1} (píxeles pixel0000..pixel0063)')
print()
print('Distribución de clases:')
for lbl, cnt in df['label'].value_counts().sort_index().items():
    print(f'  {lbl} — {LABEL_NAMES_SHORT[lbl]:6s} ({LABEL_FULL[lbl]:25s}): {cnt:5d} ({cnt/len(df)*100:.1f}%)')

In [ ]:
# Distribución de clases — gráfico
class_counts = df['label'].value_counts().sort_index()
colors = ['#e74c3c','#e67e22','#2ecc71','#3498db','#9b59b6','#1abc9c','#e91e63']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar([LABEL_NAMES_SHORT[i] for i in range(7)],
              [class_counts[i] for i in range(7)],
              color=colors, edgecolor='white', linewidth=1.5)
for bar, cnt in zip(bars, [class_counts[i] for i in range(7)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 60,
            str(cnt), ha='center', fontweight='bold', fontsize=10)
ax.set_title('Distribución de Clases — HAM10000', fontsize=14, fontweight='bold')
ax.set_xlabel('Clase Diagnóstica')
ax.set_ylabel('Número de Muestras')
ax.set_ylim(0, 7800)
plt.tight_layout()
plt.savefig('distribucion_clases.png', dpi=150, bbox_inches='tight')
plt.show()
print('Nota: El dataset presenta un fuerte desbalance de clases (nv = 66.9%).')
print('      Esto se manejará mediante pesos de muestra durante el entrenamiento.')

In [ ]:
# Visualización de ejemplos — 3 muestras por clase
feature_cols = [c for c in df.columns if c.startswith('pixel')]

fig, axes = plt.subplots(3, 7, figsize=(14, 7))
fig.suptitle('Ejemplos de Patrones por Clase (imágenes 8×8 píxeles, escala de grises)', 
             fontsize=13, fontweight='bold')

for col_idx, label in enumerate(range(7)):
    samples = df[df['label'] == label][feature_cols].values
    for row_idx in range(3):
        ax = axes[row_idx, col_idx]
        ax.imshow(samples[row_idx].reshape(8, 8).astype(np.uint8), cmap='gray', vmin=0, vmax=255)
        ax.axis('off')
        if row_idx == 0:
            ax.set_title(LABEL_NAMES_SHORT[label], fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('ejemplos_patrones.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocesamiento

In [ ]:
X = df[feature_cols].values
y = df['label'].values

# Normalización al rango [0, 1]
X_norm = X / 255.0

# División estratificada 75% / 25%
X_train, X_val, y_train, y_val = train_test_split(
    X_norm, y, test_size=0.25, random_state=42, stratify=y
)

# Pesos de muestra para compensar el desbalance de clases
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print(f'Entrenamiento: {len(X_train):,} muestras (75%)')
print(f'Validación:    {len(X_val):,} muestras (25%)')
print(f'Features:      {X_train.shape[1]} (píxeles normalizados [0,1])')

## 4. Definición y Entrenamiento de la RNA

**Arquitectura MLP con Backpropagation:**
```
Entrada (64)  →  Oculta 1 (128, ReLU)  →  Oculta 2 (64, ReLU)  →  Salida (7, Softmax)
```

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),   # 2 capas ocultas: 128 y 64 neuronas
    activation='relu',              # Activación en capas ocultas
    solver='adam',                  # Optimizador Adam
    alpha=0.0001,                   # Regularización L2
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=200,
    random_state=42,
    verbose=True,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15
)

print('Iniciando entrenamiento con pesos de clase balanceados...')
mlp.fit(X_train, y_train, sample_weight=sample_weights)
print(f'\nEntrenamiento finalizado — {mlp.n_iter_} épocas | Loss final: {mlp.loss_:.6f}')

## 5. Curva de Aprendizaje

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mlp.loss_curve_, color='#3498db', linewidth=2, label='Loss (entrenamiento)')
ax.set_title('Curva de Aprendizaje — MLP Backpropagation', fontsize=13, fontweight='bold')
ax.set_xlabel('Épocas')
ax.set_ylabel('Loss (Entropía Cruzada)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('curva_aprendizaje.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Evaluación en Conjunto de Validación

In [ ]:
y_pred = mlp.predict(X_val)
acc = accuracy_score(y_val, y_pred)

print(f'Accuracy en validación: {acc*100:.2f}%')
print(f'Error general:          {(1-acc)*100:.2f}%')
print()
target_names = [LABEL_NAMES_SHORT[i] for i in range(7)]
print(classification_report(y_val, y_pred, target_names=target_names, zero_division=0))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm, display_labels=target_names).plot(ax=ax, cmap='Blues')
ax.set_title('Matriz de Confusión — Conjunto de Validación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Tabla de Resultados por Patrón de Validación
*(Requerida por la cátedra — muestra 5 ejemplos por clase)*

In [ ]:
results = []
for lbl in range(7):
    idxs = np.where(y_val == lbl)[0][:5]
    for i in idxs:
        raw = (X_val[i] * 255).astype(int)
        entrada_resumen = f'[{raw[0]}, {raw[1]}, {raw[2]}, ..., {raw[63]}]'
        results.append({
            'Patrón #': i,
            'Entrada (p0, p1, p2, ..., p63)': entrada_resumen,
            'Salida RNA': LABEL_NAMES_SHORT[y_pred[i]],
            'Salida Esperada': LABEL_NAMES_SHORT[y_val[i]],
            'Correcto': 'SI' if y_pred[i] == y_val[i] else 'NO'
        })

results_df = pd.DataFrame(results)
print(f'Correctos en muestra: {(results_df["Correcto"]=="SI").sum()} / {len(results_df)}')
results_df

In [ ]:
# Exportar tabla completa de validación a CSV
full_results = pd.DataFrame({
    'Patron': range(len(y_val)),
    'Salida_RNA': [LABEL_NAMES_SHORT[p] for p in y_pred],
    'Salida_Esperada': [LABEL_NAMES_SHORT[e] for e in y_val],
    'Correcto': ['SI' if p == e else 'NO' for p, e in zip(y_pred, y_val)]
})
full_results.to_csv('resultados_validacion.csv', index=False, encoding='utf-8')
print(f'Exportado: resultados_validacion.csv ({len(full_results)} patrones)')
print(f'Correctos totales: {(full_results["Correcto"]=="SI").sum()} / {len(full_results)}')

## 8. Resumen Final

In [ ]:
print('=' * 55)
print('   RESUMEN — RNA CLASIFICACIÓN CÁNCER DE PIEL')
print('=' * 55)
print(f'Dataset:              HAM10000 (hmnist_8_8_L.csv)')
print(f'Total muestras:       {len(df):,}')
print(f'Entrenamiento:        {len(X_train):,} muestras (75%)')
print(f'Validación:           {len(X_val):,} muestras (25%)')
print(f'Arquitectura MLP:     64 → 128 (ReLU) → 64 (ReLU) → 7 (Softmax)')
print(f'Optimizador:          Adam (lr=0.001)')
print(f'Balanceo de clases:   sample_weight=balanced')
print(f'Épocas ejecutadas:    {mlp.n_iter_}')
print(f'Loss final:           {mlp.loss_:.6f}')
print(f'Accuracy validación:  {acc*100:.2f}%')
print('=' * 55)